Other models that can be considered include the VGG16, ResNet50, MobileNetV2 etc. 

The custom CNN (Conv2D) leanrs image features from scratch. MobileNetV2, on the other hand, uses features learned from a much larger dataset called ImageNet. For example, MobileNetV2 has already learned to recognise visual patterns such as:

- Edges and corners
- Textures
- shapes
- more complex visual structures

We reuse these learned features and add a new classification layer that learns to distinguish between cats and Dogs

In [1]:
import tensorflow as tf

### Load the pretrained MobileNetV2 model

- include_top = False: The pretrained model was originally designed to classify 1000 ImageNet categories. The original classification layers are called the top of the network. Setting the top to false removes these original classification layers because we want to create our own classifier for just two categories namely: Cat and Dog.
- weights = "imagenet": loads weights eanred from training on ImageNet. These weights contain previously learned image features. Instead of starting with randomly initialized weights, we start with a network that already understands many patterns.
- input_shape: Specifies the expected image dimensions

In [2]:
#keras.applications.MobileNetV2 loads the architecture from Keras library
base_model = tf.keras.applications.MobileNetV2(
    input_shape = (224, 224, 3),
    include_top=False,
    weights="imagenet"
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


### Freeze the pretrained Layers

Freezing the pretrained layers prevents the MobileNetV2 layers from being updated during the initial training process. Why do we do this? imagine MobileNetV2 has already learned to recognise edges, shapes and textures from millions of images. We want to preserve this knowledge rather than immediately chaning all its weights when training on our smaller cats vs Dogs dataset. So what happens:

- MobileNetV2 extracts the image features
- the newly added classification layer learns to distinguish cats from dogs

This approach is known as feature extraction which is a form of transfer learning

In [3]:
base_model.trainable = False

### Define the Model Input

This will create the input layer of our new classification model. Note that we are creating a new model around MobileNetV2 rathen than modifying the original ImageNet classification model

In [4]:
#This defines the expected shape of an image that will enter the neural network

inputs = tf.keras.Input(shape=(224, 224, 3))

### Apply Data Augmentation

The first stage applies that data augmentation pipeline that we previously created under the simple Conv2D model

In [6]:
# Import Data Augmentation
from tensorflow.keras.layers import (
    RandomFlip,
    RandomRotation,
    RandomZoom,
    RandomContrast,
    RandomTranslation
)


data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1)
])

# this is a transformation to the inputs defined above. 
x = data_augmentation(inputs)

### Apply MobileNetV2 preprocessing

In [7]:
x = tf.keras.layers.Rescaling(
    scale=1./127.5,
    offset=-1
)(x)

### Pass Images through MobileNetV2

This passes the augmented and preprocessed images through the pretrained MobileNetV2 network. MobileNetV2 extracts important image features. For example, it may learn feature representation associated with:

- Fur textures
- animal ears
- facial structures
- body shapes
- other visual patterns

the training = False ensures that the pretrained network operates in inference mode. This is important because the network contains batch normalization layers. We want their pretrained statistics to remain stable while we train our new classification layers

In [8]:
x = base_model(x, training=False)

### Apply Global Average Pooling

The mobileNetV2 produces feature maps and not a single classification result. For instance (7, 7, 1280). (feature-map height, feature-map width, number of feature channels). The GlobalAveragePooling2D() calculates the average value of each feature channel across its spatial dimensions, thereby transforming the MobileNetV2 feature maps to (1280, ). In other words, It converts the feature maps into a one-dimensional feature vector that can be passed to the Dense classification layer. In addition, it reduces the number of parameters compared with using Flatten() on the entire feature map

In [9]:
x = tf.keras.layers.GlobalAveragePooling2D()(x)

### Add Classification Layer

We are adding a dense output layer containing one neuron

In [11]:
outputs = tf.keras.layers.Dense(
    1,
    activation="sigmoid"
)(x)

### Construct the Complete Model

Combines all the layers into one complete Keras model.

The architecture now contains:
1. Input layer.
2. Data augmentation.
3. Image preprocessing.
4. Pretrained MobileNetV2 feature extractor.
5. Global Average Pooling.
6. Dense sigmoid output layer.

We store this model in the variable mobilenet_model.

In [12]:
mobilenet_model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs
)

In [14]:
# Inspect the architecture using

mobilenet_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

### Compile the Model

In [15]:
mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

### Define Early Stopping

In [16]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

### Train MobileNetV2

In [17]:
mobilenet_history = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stopping]
)

NameError: name 'train_ds' is not defined